# 07 — Sky Camera Labelling Tool

On every image load the tool automatically applies the **antenna ignore mask** and
marks the **sun disk** as IGNORE based on the filename timestamp.

---

## Buttons (bottom of window)
| Group | Buttons |
|-------|---------|
| Mode | `SAM`  `Lasso`  `Brush` |
| Label | `☁ Cloud`  `Sky`  `Ignore` |
| Navigate | `◀ Prev`  `Next ▶`  `Skip` (no save) |
| Actions | `Save`  `Reset` |

---

## SAM mode (default)
| Input | Action |
|-------|--------|
| Left-click | Segment → **Cloud** preview |
| Right-click | Segment → **Sky** preview |
| Middle-click | Segment → **Ignore** preview |
| `Tab` | Cycle 3 candidates (tight / medium / loose) |
| `a` | Accept current preview |
| `Esc` | Cancel preview / return to SAM mode |
| `o` | Toggle **overwrite mode** — SAM overwrites existing labels (off by default) |

## Lasso mode (`f` key or button)
| Input | Action |
|-------|--------|
| Hold L + drag | Draw freehand outline |
| Release | Fill **inside** with active label |
| `x` | Fill **outside** with active label |
| `c` / `s` / `i` | Set active label → Cloud / Sky / Ignore |

## Brush mode (`b` key or button)
| Input | Action |
|-------|--------|
| L-drag | Paint active label |
| `c` / `s` / `i` | Set active label |
| `+` / `-` | Brush size |

## Global keys
| Key | Action |
|-----|--------|
| `S` | Save current mask |
| `n` / `p` | Next / previous image (auto-saves) |
| `N` | Next image **without** saving |
| `z` | Undo |
| `r` | Reset mask (re-applies antenna + sun ignore) |
| `q` | Quit (auto-saves) |

In [1]:
%load_ext autoreload
%autoreload 2

import sys, logging
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s | %(message)s',
                    datefmt='%H:%M:%S')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skycamera.config import CX, CY, R, MASKS_MANUAL_DIR, RAW_DIR
from skycamera.io import load_image
from skycamera.preprocessing import build_circular_mask
from skycamera.sam_labelling import (
    load_sam2, batch_pseudolabel, run_batch_pseudolabel,
    SAMLabellingTool, SAM2_CHECKPOINT,
    create_default_ignore_mask, DEFAULT_IGNORE_FILENAME,
)
from skycamera.labelling import load_existing_mask, LABEL_CLOUD, LABEL_SKY, LABEL_UNLABELLED
from skycamera.sun import sun_altaz, parse_timestamp

RAW_ROOT  = RAW_DIR
MASKS_DIR = Path('..') / 'data' / 'masks_manual'
MASKS_DIR.mkdir(parents=True, exist_ok=True)
DEFAULT_IGNORE_PATH = MASKS_DIR / DEFAULT_IGNORE_FILENAME

_s = load_image(RAW_ROOT / '2024-01-15' / '2024_01_15__12_04_51.jpg')
dome_mask = build_circular_mask(_s.shape[0], _s.shape[1], CX, CY, R)

# Check pysolar is available
try:
    import pysolar
    print('pysolar OK')
except ImportError:
    print('WARNING: pysolar not installed — sun position unavailable')
    print('Fix: pip install pysolar')

print(f'Setup complete  |  checkpoint: {SAM2_CHECKPOINT.name}')
print(f'Default ignore mask: {"EXISTS" if DEFAULT_IGNORE_PATH.exists() else "not created yet — run cell below"}')

pysolar OK
Setup complete  |  checkpoint: sam2.1_hiera_small.pt
Default ignore mask: EXISTS


In [2]:
%matplotlib tk
import importlib
import matplotlib.patches as mpatches
import matplotlib.widgets as mwidgets
import skycamera.config as _cfg
import skycamera.sun as _sun

# Pick a sunny midday image
test_img_path = RAW_ROOT / '2024-06-15' / '2024_06_15__12_00_31.jpg'
test_img = load_image(test_img_path)

# Parse timestamp once — doesn't change
from skycamera.sun import parse_timestamp, sun_altaz
dt       = parse_timestamp(test_img_path)
alt, az  = sun_altaz(dt)
print(f'Sun altitude: {alt:.1f}°   azimuth (N-based): {az:.1f}°')

fig, ax = plt.subplots(figsize=(9, 9))
plt.subplots_adjust(bottom=0.15)
ax.imshow(test_img)
ax.axis('off')
_patches = []

def redraw(_event=None):
    importlib.reload(_cfg)
    importlib.reload(_sun)
    for p in _patches:
        try: p.remove()
        except Exception: pass
    _patches.clear()

    px   = _sun.altaz_to_pixel(alt, az)
    r_px = int(_cfg.R * _cfg.SUN_IGNORE_RADIUS_DEG / 90.0 * _cfg.CAMERA_PROJECTION_SCALE)

    if px:
        c = mpatches.Circle(px, r_px, fill=False, edgecolor='orange', linewidth=2)
        ax.add_patch(c)
        cross, = ax.plot(*px, '+', color='orange', markersize=20, markeredgewidth=2)
        _patches.extend([c, cross])

    ax.set_title(
        f'alt={alt:.1f}°  az={az:.1f}°  →  pixel={px}\n'
        f'CAMERA_NORTH_OFFSET_DEG={_cfg.CAMERA_NORTH_OFFSET_DEG}°  '
        f'CAMERA_PROJECTION_SCALE={_cfg.CAMERA_PROJECTION_SCALE:.2f}  '
        f'SUN_IGNORE_RADIUS_DEG={_cfg.SUN_IGNORE_RADIUS_DEG}°\n'
        'Edit config.py and click Refresh',
        fontsize=9,
    )
    fig.canvas.draw_idle()

ax_btn = fig.add_axes([0.38, 0.04, 0.24, 0.06])
btn = mwidgets.Button(ax_btn, '↺  Refresh from config.py', color='#333333', hovercolor='#555555')
btn.label.set_color('white')
btn.on_clicked(redraw)

redraw()
plt.show()

Sun altitude: 56.9°   azimuth (N-based): 37.1°


16:25:54 | Mask saved: ..\data\masks_manual\2024_01_15__15_34_31_GT.png
16:25:54 | For numpy array image, we assume (HxWxC) format
16:25:54 | Computing image embeddings for the provided image...
16:25:56 | Image embeddings computed.
16:26:06 | Mask saved: ..\data\masks_manual\2024_01_15__16_37_00_GT.png
Traceback (most recent call last):
  File "c:\Users\szymo\anaconda3\envs\geo\Lib\site-packages\matplotlib\cbook.py", line 361, in process
    func(*args, **kwargs)
  File "c:\Users\szymo\anaconda3\envs\geo\Lib\site-packages\matplotlib\widgets.py", line 244, in <lambda>
    return self._observers.connect('clicked', lambda event: func(event))
                                                            ^^^^^^^^^^^
  File "d:\MOJE\DATA_SCIENCE\SKYCAMERA\skycamera\notebooks\..\src\skycamera\sam_labelling.py", line 1032, in <lambda>
    "Next ▶":   lambda _: self._btn_nav(+1),
                          ^^^^^^^^^^^^^^^^^
  File "d:\MOJE\DATA_SCIENCE\SKYCAMERA\skycamera\notebooks\..\src\skycame

## 2. Sun position calibration (run once to verify)

Edit `CAMERA_NORTH_OFFSET_DEG`, `CAMERA_PROJECTION_SCALE`, `SUN_IGNORE_RADIUS_DEG` in [config.py](../src/skycamera/config.py), then click **↺ Refresh** in the plot window until the circle sits on the actual sun.

## 3. Load SAM 2 model

## 4. Create / edit default ignore mask (run once)

Paint fixed structures visible in every image (antennas, cables).
Left-click = SAM ignore, right-drag = erase, `s` = save & quit.

In [3]:
predictor = load_sam2()
print('SAM 2 predictor ready.')

16:25:06 | Loaded checkpoint sucessfully
16:25:06 | SAM2 loaded: sam2.1_hiera_small.pt on cpu


SAM 2 predictor ready.


In [ ]:
%matplotlib tk

reference_img = RAW_ROOT / '2024-01-15' / '2024_01_15__12_04_51.jpg'

create_default_ignore_mask(
    reference_image_path = reference_img,
    masks_dir            = MASKS_DIR,
    predictor            = predictor,
    dome_mask            = dome_mask,
    brush_radius         = 15,
)
print(f'Default ignore mask ready: {DEFAULT_IGNORE_PATH}')

## 5. Interactive labelling (Mode A)

See controls table at the top of this notebook.

In [4]:
%matplotlib tk
import random

all_images = sorted(RAW_ROOT.rglob('*.jpg'))
already_labelled = {p.stem.replace('_GT', '') for p in MASKS_DIR.glob('*_GT.png')}
unlabelled = [p for p in all_images if p.stem not in already_labelled]

n_sample = 30
image_paths = random.sample(unlabelled, min(n_sample, len(unlabelled)))
image_paths.sort()

print(f'Total: {len(all_images)}  |  labelled: {len(already_labelled)}  |  queued: {len(image_paths)}')

tool = SAMLabellingTool(
    image_paths         = image_paths,
    masks_dir           = MASKS_DIR,
    dome_mask           = dome_mask,
    predictor           = predictor,
    brush_radius        = 15,
    default_ignore_path = DEFAULT_IGNORE_PATH,
)
tool.run()

Total: 3328  |  labelled: 0  |  queued: 30


16:25:09 | For numpy array image, we assume (HxWxC) format
16:25:10 | Computing image embeddings for the provided image...
16:25:13 | Image embeddings computed.


## 4. Mode B — Batch pseudo-labelling

Runs automatically on all images — no human interaction needed.
Use these masks as training data for the CNN (notebook 03).

Adjust `grid_n` and `brightness_threshold` if results are under/over-segmenting.

In [ ]:
# Collect images to pseudo-label (one per month for a quick run)
batch_paths = []
for month_dir in sorted(RAW_ROOT.glob('2024-*')):
    imgs = sorted(month_dir.glob('*.jpg'))
    if imgs:
        batch_paths.append(imgs[0])   # first image of each month

print(f'Images to pseudo-label: {len(batch_paths)}')
print('Running batch SAM2 pseudo-labelling...')

saved = run_batch_pseudolabel(
    image_paths         = batch_paths,
    masks_dir           = MASKS_DIR,
    dome_mask           = dome_mask,
    grid_n              = 10,
    brightness_threshold = 0.55,
    overwrite           = False,
)
print(f'Saved {len(saved)} masks to {MASKS_DIR}')

Images to pseudo-label: 12
Running batch SAM2 pseudo-labelling...


10:20:42 | Loaded checkpoint sucessfully
10:20:42 | SAM2 loaded: sam2.1_hiera_small.pt on cpu
10:20:42 | For numpy array image, we assume (HxWxC) format
10:20:42 | Computing image embeddings for the provided image...
10:20:45 | Image embeddings computed.
10:20:52 | Mask saved: ..\data\masks_manual\2024_01_15__12_04_51_GT.png
10:20:52 | [1/12] 2024_01_15__12_04_51.jpg -> CF=0.994 saved 2024_01_15__12_04_51_GT.png
10:20:52 | For numpy array image, we assume (HxWxC) format
10:20:52 | Computing image embeddings for the provided image...
10:20:56 | Image embeddings computed.
10:21:08 | Mask saved: ..\data\masks_manual\2024_02_15__12_02_10_GT.png
10:21:08 | [2/12] 2024_02_15__12_02_10.jpg -> CF=0.986 saved 2024_02_15__12_02_10_GT.png
10:21:08 | For numpy array image, we assume (HxWxC) format
10:21:08 | Computing image embeddings for the provided image...
10:21:12 | Image embeddings computed.
10:21:17 | Mask saved: ..\data\masks_manual\2024_03_15__12_05_02_GT.png
10:21:17 | [3/12] 2024_03_15_

## 5. Labelling summary

In [ ]:
log_path = MASKS_DIR / 'labelling_log.csv'
gt_pngs  = sorted(MASKS_DIR.glob('*_GT.png')) if MASKS_DIR.exists() else []

print(f'Total masks saved: {len(gt_pngs)}')

if log_path.exists():
    df_log = pd.read_csv(log_path)
    print(f'Log entries: {len(df_log)}')
    print(f'Mean CF: {pd.to_numeric(df_log["cf_estimate"], errors="coerce").mean():.3f}')
    display(df_log.tail(10))

if gt_pngs:
    # Thumbnail gallery
    n = min(len(gt_pngs), 12)
    fig, axes = plt.subplots(2, n, figsize=(2.5 * n, 5))
    for col, mask_path in enumerate(gt_pngs[:n]):
        stem  = mask_path.stem.replace('_GT', '')
        cands = list(RAW_ROOT.rglob(stem + '.jpg'))
        if cands:
            axes[0][col].imshow(load_image(cands[0]))
        else:
            axes[0][col].set_facecolor('black')
        axes[0][col].axis('off')
        axes[0][col].set_title(stem[-10:], fontsize=6)

        mask = load_existing_mask(mask_path)
        vis  = np.zeros((*mask.shape, 3), dtype=np.uint8)
        vis[mask == LABEL_CLOUD] = [220, 220, 220]
        vis[mask == LABEL_SKY]   = [30, 120, 200]
        cf = _compute_cf(mask)
        axes[1][col].imshow(vis)
        axes[1][col].axis('off')
        axes[1][col].set_title(f'CF={cf:.2f}', fontsize=6)

    fig.suptitle('SAM2 generated masks', fontsize=10)
    fig.tight_layout()
    plt.savefig('../outputs/plots/sam2_mask_gallery.png', bbox_inches='tight', dpi=80)
    plt.show()